# Rank feature extraction (3 s windows)

Build a BreathMetrics feature matrix from cagemate-rank respiration H5 files (BLA_ChR pilot + ECG cohort1), using non-overlapping 3 s windows.


## Imports and environment setup


In [1]:
# Imports and environment setup
# --- Core Python ---
import os
import sys
from pathlib import Path
import h5py
import re

# Ensure notebook-local helper modules are importable
_NB_DIR = Path.cwd()
print(f"currently in {_NB_DIR}")
if (_NB_DIR / "resp_helper_functions.py").exists():
    print("helper functions exists in working directory, adding to sys.path")
    sys.path.insert(0, str(_NB_DIR))
else:
    # Fallback when notebook is launched from workspace root
    _NB_DIR = Path("notebooks/thomas_notebooks/resp_classification").resolve()
    if (_NB_DIR / "resp_helper_functions.py").exists():
        sys.path.insert(0, str(_NB_DIR))
    print("Helper functions didn't exist in working directory, resolving and adding to sys.path")

# --- Numerical & data analysis ---
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Signal processing ---
from scipy.signal import butter, filtfilt, resample_poly, find_peaks

# --- Statistics ---
from scipy.stats import wilcoxon

# --- Machine learning & metrics ---
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# --- Specialized neurophysiology tools ---
import neurokit2 as nk

# Shared signal + breathmetrics utilities only (rank pipeline lives in this notebook)
from resp_helper_functions import (
    fit_bm_session,
    sample_random_nonoverlapping_windows,
    extract_features_from_windows,
    build_rank_cagemate_resp_paths_from_dir,
    build_rank_cagemate_window_feature_matrix,
)

from math import gcd


# --- Pandas display settings ---
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)


currently in c:\Users\thoma\Code\ResearchCode\respiratory_pilot
Helper functions didn't exist in working directory, resolving and adding to sys.path
Libraries loaded successfully


## Data path — BLA_ChR cagemate-rank H5 outputs


In [4]:
# Data path — BLA_ChR cagemate-rank H5 outputs
cagemate_rank_h5s = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1 (SS)\BLA_resp_ChR_cm_hc\h5_outputs"

## List H5 files in the cagemate-rank directory


In [5]:
# List H5 files in the cagemate-rank directory
_h5_dir = Path(cagemate_rank_h5s)
rank_h5_files = sorted(p.name for p in _h5_dir.iterdir() if p.is_file())
print(f"{len(rank_h5_files)} files in {cagemate_rank_h5s}:\n")
for name in rank_h5_files:
    print(name)

25 files in C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1 (SS)\BLA_resp_ChR_cm_hc\h5_outputs:

1_1_i_cm_1_2_d_20260106_134350.h5
1_1_i_cm_1_3_s_20260107_163611.h5
1_2_d_cm_1_3_s_20260108_114416.h5
1_3_s_1_2_d_20260107_165340.h5
1_3_s_cm_1_1_i_20260106_122823.h5
2_1_d_cm_2.3_i_20260106_105109.h5
2_1_d_cm_2_2_s_20260108_112426.h5
2_2_s_cm_2_1_d_20260108_120453.h5
2_2_s_cm_2_1_d_20260108_120453.rec.h5
2_2_s_cm_2_3_i_20260106_132549.h5
3_1_i_cm_3_2_d_20260106_115157.h5
3_1_i_cm_3_3_s_20260107_123230.h5
3_2_d_cm_3_3_s_20260107_140924.h5
3_3_s_cm_3_1_i_20260106_124546.h5
3_3_s_cm_3_2_d_20260107_171330.h5
4_1_s_cm_4_2_d_20260107_160439.h5
4_2_d_cm_4_1_s_20260107_150836.h5
4_3_d_cm_4_1_s_20260107_113120.h5
5_1_i_cm_5_2_d_20260106_120946.h5
5_1_i_cm_5_3_s_20260107_132309.h5
6_2_d_cm_6_3_s_20260107_154939.h5
7_1_d_cm_7_2_s_20260106_111330.h5
7_2_s_cm_7_1_d_20260106_130431.h5
8_1_s_cm_8_2_d_20260107_152812.h5
8_2_s_cm_8_1_d_20260106_113241.h5


### AIM 1 Cagemate Respiration H5s

In [ ]:
CM_data_path = "C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs"

## Filename rename map (raw → standardized SUB/DOM names)


In [6]:
# Filename rename map (raw → standardized SUB/DOM names)
rename_map = {
    "1_1_d_cm_1_2_i_20260106_134350.h5": "1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5",
    "1_1_i_cm_1_3_s_20260107_163611.h5": "1_1_2_i_cm_1_3_s_20260107_163611_DOM.h5",
    "1_2_d_cm_1_3_s_20260108_114416.h5": "1_2_2_d_cm_1_3_s_20260108_114416_DOM.h5",
    "1_3_s_cm_1_1_i_20260106_122823.h5": "1_3_2_s_cm_1_1_i_20260106_122823_DOM.h5",
    "1_3_s_1_2_d_20260107_165340.h5": "1_3_2_s_cm_1_2_d_20260107_165340_SUB.h5",
    "2_1_d_cm_2.3_i_20260106_105109.h5": "2_1_2_d_cm_2.3_i_20260106_105109_SUB.h5",
    "2_1_d_cm_2_2_s_20260108_112426.h5": "2_1_2_d_cm_2_2_s_20260108_112426_DOM.h5",
    "2_2_s_cm_2_1_d_20260108_120453.h5": "2_2_2_s_cm_2_1_d_20260108_120453_SUB.h5",
    "2_2_s_cm_2_3_i_20260106_132549.h5": "2_2_2_s_cm_2_3_i_20260106_132549_DOM.h5",
    "3_1_i_cm_3_2_d_20260106_115157.h5": "3_1_2_i_cm_3_2_d_20260106_115157_SUB.h5",
    "3_1_i_cm_3_3_s_20260107_123230.h5": "3_1_2_i_cm_3_3_s_20260107_123230_DOM.h5",
    "3_2_d_cm_3_3_s_20260107_140924.h5": "3_2_2_d_cm_3_3_s_20260107_140924_DOM.h5",
    "3_3_s_cm_3_1_i_20260106_124546.h5": "3_3_2_s_cm_3_1_i_20260106_124546_DOM.h5",
    "3_3_s_cm_3_2_d_20260107_171330.h5": "3_3_2_s_cm_3_2_d_20260107_171330_SUB.h5",
    "4_1_s_cm_4_2_d_20260107_160439.h5": "4_1_2_s_cm_4_2_d_20260107_160439_SUB.h5",
    "4_2_d_cm_4_1_s_20260107_150836.h5": "4_2_2_d_cm_4_1_s_20260107_150836_DOM.h5",
    "4_3_d_cm_4_1_s_20260107_113120.h5": "4_3_2_d_cm_4_1_s_20260107_113120_DOM.h5",
    "5_1_i_cm_5_2_d_20260106_120946.h5": "5_1_2_i_cm_5_2_d_20260106_120946_SUB.h5",
    "5_1_i_cm_5_3_s_20260107_132309.h5": "5_1_2_i_cm_5_3_s_20260107_132309_DOM.h5",
    "6_2_d_cm_6_3_s_20260107_154939.h5": "6_2_2_d_cm_6_3_s_20260107_154939_DOM.h5",
    "7_1_d_cm_7_2_s_20260106_111330.h5": "7_1_2_d_cm_7_2_s_20260106_111330_DOM.h5",
    "7_2_s_cm_7_1_d_20260106_130431.h5": "7_2_2_s_cm_7_1_d_20260106_130431_SUB.h5",
    "8_1_s_cm_8_2_d_20260107_152812.h5": "8_1_2_s_cm_8_2_d_20260107_152812_SUB.h5",
    "8_2_s_cm_8_1_d_20260106_113241.h5": "8_2_2_s_cm_8_1_d_20260106_113241_SUB.h5",
}

## Cohort1 cagemate paths, rank map, and `build_cagemate_rank_feature_matrix_from_paths`


In [7]:
# Cohort1 cagemate paths, rank map, and build_cagemate_rank_feature_matrix_from_paths
# --- Added cohort1 CM recordings that use an explicit rank map ---
cohort1_cm_h5s = Path(r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs")

cohort1_rank_map = {
    "1_1": "Subordinate",
    "1_2": "Dominant",
    "2_3": "Subordinate",
    "2_4": "Dominant",
    "3_5": "Subordinate",
    "3_6": "Dominant",
    "4_7": "Subordinate",
    "4_8": "Dominant",
}

cohort1_pair_groups = [
    ["1_1", "1_2"],
    ["2_3", "2_4"],
    ["3_5", "3_6"],
    ["4_7", "4_8"],
]

resp_paths_cm_cohort1 = {
    "CM_1_1_d1_2": str(cohort1_cm_h5s / "CM_s1_1_d1_2_20250623_111352_merged.h5"),
    "CM_1_2_sub1_1": str(cohort1_cm_h5s / "CM_s1_2_sub1_1_20250623_133932_merged.h5"),
    "CM_2_3_d2_4": str(cohort1_cm_h5s / "CM_s2_3_d2_4_20250623_151153_merged.h5"),
    "CM_2_4_sub2_3": str(cohort1_cm_h5s / "CM_s2_4_sub2_3_20250623_143348_merged.h5"),
    "CM_3_5_d3_6": str(cohort1_cm_h5s / "CM_s3_5_d3_6_20250623_170708_merged.h5"),
    "CM_3_6_sub3_5": str(cohort1_cm_h5s / "CM_s3_6_sub3_5_20250623_174348_merged.h5"),
    "CM_4_7_d4_8": str(cohort1_cm_h5s / "CM_s4_7_d4_8_20250623_193718_merged.h5"),
    "CM_4_8_sub4_7": str(cohort1_cm_h5s / "CM_s4_8_sub4_7_20250623_182649_merged.h5"),
}

print(f"Added {len(resp_paths_cm_cohort1)} cohort1 CM recordings from {cohort1_cm_h5s}")


def normalize_rank_label(rank_value):
    if pd.isna(rank_value):
        return np.nan

    rank_value = str(rank_value).strip().lower()
    if rank_value in {"dominant", "dom"}:
        return "DOM"
    if rank_value in {"subordinate", "sub"}:
        return "SUB"
    return rank_value.upper()


def parse_cohort1_cm_metadata(raw_name, rank_map):
    """Parse cohort1 cagemate filenames.

    Supports both dominant/intermediate/subordinate short codes (d/i/s)
    and the explicit `sub` token used in some filenames.

    Examples:
      - CM_s1_1_d1_2_20250623_111352_merged.h5
      - CM_s1_2_sub1_1_20250623_133932_merged.h5
    """

    raw_base = os.path.splitext(str(raw_name))[0]

    pattern = r"^CM_s(\d+_\d+)_([a-z]+)(\d+_\d+)_(\d{8})_(\d{6})_merged$"
    m = re.match(pattern, raw_base, flags=re.IGNORECASE)
    if not m:
        return None

    subject_id = str(m.group(1))
    subject_rank_code = str(m.group(2)).lower()
    cagemate_id = str(m.group(3))
    date_str = str(m.group(4))
    time_str = str(m.group(5))

    subject_rank_name = rank_map.get(subject_id, np.nan)
    cagemate_rank_name = rank_map.get(cagemate_id, np.nan)
    rank_label = normalize_rank_label(subject_rank_name)

    trial_key = f"{subject_id}_cm_{cagemate_id}_{date_str}_{time_str}"

    return {
        "Recording": raw_base,
        "StandardizedRecording": raw_base,
        "PairID": subject_id.split("_")[0],
        "Subject": subject_id,
        "Cagemate": cagemate_id,
        "SubjectRankCode": subject_rank_code,
        "CagemateRankCode": np.nan,
        "SubjectRankName": subject_rank_name,
        "CagemateRankName": cagemate_rank_name,
        "RankLabel": rank_label,
        "TrialKey": trial_key,
    }


def build_cagemate_rank_feature_matrix_from_paths(
    file_map,
    rank_map,
    data_type="rodentAirflow",
    target_srate=400,
    window_sec=3.0,
    n_windows_per_session=100,
    min_breaths_per_window=5,
    random_state=42,
):
    all_rows = []

    for raw_name, h5_path in sorted(file_map.items()):
        # ``file_map`` keys can be short trial IDs (e.g. "CM_1_1_d1_2") or full
        # H5 basenames. Always parse using the actual filename on disk so the
        # regex sees the expected "CM_s..." pattern.
        basename = os.path.basename(str(h5_path))
        meta_info = parse_cohort1_cm_metadata(basename, rank_map)
        if meta_info is None:
            print(f"  ⚠️ Skipping {raw_name}: unexpected filename pattern (basename={basename})")
            continue

        print(f"\n[Rank cohort1] {basename}")

        signal, time, fs, load_meta = load_clean_resp_signal_cagemate_rank(
            h5_path,
            target_rate=target_srate,
        )

        if signal is None:
            print("  ❌ Load failed")
            continue

        if len(signal) == 0 or len(time) == 0:
            print("  ❌ Empty signal after preprocessing")
            continue

        bm = fit_bm_session(signal, fs, data_type=data_type)
        if bm is None:
            print("  ❌ BreathMetrics fit failed")
            continue

        n_breaths_total = len(getattr(bm, "inhaleOnsets", []))
        print(f"  ✓ {n_breaths_total} breaths detected in full session")

        windows = sample_random_nonoverlapping_windows(
            time=time,
            signal=signal,
            window_dur=window_sec,
            n_windows=n_windows_per_session,
            seed=random_state,
            allow_partial_if_short=False,
        )

        if len(windows) == 0:
            print("  ⚠️ No usable full windows")
            continue

        window_rows = extract_features_from_windows(
            bm=bm,
            windows=windows,
            min_breaths_per_window=min_breaths_per_window,
        )

        if len(window_rows) == 0:
            print("  ⚠️ No usable windows after min-breath filtering")
            continue

        print(f"  ✓ {len(window_rows)} usable windows")

        session_duration_sec = float(time[-1] - time[0]) if len(time) > 1 else np.nan

        for row in window_rows:
            row.update({
                "Trial": meta_info["TrialKey"],
                "Recording": meta_info["Recording"],
                "StandardizedRecording": meta_info["StandardizedRecording"],
                "Subject": meta_info["Subject"],
                "Cagemate": meta_info["Cagemate"],
                "PairID": meta_info["PairID"],
                "SubjectRankCode": meta_info["SubjectRankCode"],
                "CagemateRankCode": meta_info["CagemateRankCode"],
                "SubjectRankName": meta_info["SubjectRankName"],
                "CagemateRankName": meta_info["CagemateRankName"],
                "Condition": meta_info["RankLabel"],
                "RankLabel": meta_info["RankLabel"],
                "Type": "CagemateRank",
                "SourceCohort": "ECG_cohort1",
                "session_duration_sec": session_duration_sec,
                "n_breaths_total": n_breaths_total,
                "WindowSec": window_sec,
                "TargetSRate": target_srate,
                "MinBreathsPerWindow": min_breaths_per_window,
            })
            all_rows.append(row)

    if not all_rows:
        print("\n⚠️ No rows collected for cohort1.")
        return pd.DataFrame()

    cohort1_df = pd.DataFrame(all_rows).reset_index(drop=True)

    print(f"\n✅ Cohort1 cagemate rank feature matrix: {len(cohort1_df)} rows × {len(cohort1_df.columns)} cols")
    print(f"   Recordings : {cohort1_df['Recording'].nunique()}")
    print(f"   Subjects   : {cohort1_df['Subject'].nunique()}")
    print(f"   Cagemates  : {cohort1_df['Cagemate'].nunique()}")
    print(f"   Pairs      : {cohort1_df['PairID'].nunique()}")
    print(f"   Rank dist  :\n{cohort1_df['RankLabel'].value_counts(dropna=False)}")

    return cohort1_df

Added 8 cohort1 CM recordings from C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs


## Inspect example H5 file structure


In [9]:
# Inspect example H5 file structure
import h5py

h5_path = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1 (SS)\BLA_resp_ChR_cm_hc\h5_outputs\1_1_i_cm_1_2_d_20260106_134350.h5"

def print_h5_structure(name, obj):
    print(name)

with h5py.File(h5_path, "r") as f:
    print("=== TOP LEVEL KEYS ===")
    for key in f.keys():
        print(key)

    print("\n=== FULL STRUCTURE ===")
    f.visititems(print_h5_structure)

=== TOP LEVEL KEYS ===
analog
dio
resp_clean
time

=== FULL STRUCTURE ===
analog
analog/1_1_d_cm_1_2_i_20260106_134350.analog_ECU_Ain1
analog/1_1_d_cm_1_2_i_20260106_134350.timestamps
dio
resp_clean
resp_clean/signal
resp_clean/time
time
time/1_1_d_cm_1_2_i_20260106_134350.timestamps


## `load_clean_resp_signal_cagemate_rank` — load and resample respiration


In [10]:
# load_clean_resp_signal_cagemate_rank — load and resample respiration
import h5py
import numpy as np
from scipy.signal import resample_poly
from math import gcd  # <-- this was missing


def load_clean_resp_signal_cagemate_rank(h5_path, target_rate=400):
    """
    Load cleaned respiration signal from the cagemate-rank H5 files.

    Expected H5 structure:
        resp_clean/
            signal
            time
    """
    try:
        with h5py.File(h5_path, "r") as f:
            if "resp_clean" not in f:
                raise KeyError("Missing group 'resp_clean'")
            if "signal" not in f["resp_clean"]:
                raise KeyError("Missing dataset 'resp_clean/signal'")
            if "time" not in f["resp_clean"]:
                raise KeyError("Missing dataset 'resp_clean/time'")

            raw_signal = np.asarray(f["resp_clean"]["signal"][:]).squeeze()
            raw_time = np.asarray(f["resp_clean"]["time"][:]).squeeze()

        if raw_signal.ndim != 1:
            raise ValueError(f"resp_clean/signal is not 1D. shape={raw_signal.shape}")
        if raw_time.ndim != 1:
            raise ValueError(f"resp_clean/time is not 1D. shape={raw_time.shape}")
        if len(raw_signal) == 0 or len(raw_time) == 0:
            raise ValueError("Signal or time array is empty")

        # Trim to shortest if slight mismatch
        if len(raw_signal) != len(raw_time):
            min_len = min(len(raw_signal), len(raw_time))
            print(
                f"Warning: signal/time length mismatch in {h5_path}. "
                f"Trimming from signal={len(raw_signal)}, time={len(raw_time)} to {min_len}."
            )
            raw_signal = raw_signal[:min_len]
            raw_time = raw_time[:min_len]

        # Estimate original sampling rate from time vector
        dt = np.diff(raw_time)
        dt = dt[np.isfinite(dt)]

        if len(dt) == 0:
            raise ValueError("Could not estimate sampling rate from time vector")

        median_dt = np.median(dt)
        if median_dt <= 0:
            raise ValueError(f"Non-positive median dt detected: {median_dt}")

        original_fs = 1.0 / median_dt

        # Resample if needed
        if target_rate is not None and not np.isclose(original_fs, target_rate, rtol=1e-3):
            target_rate_int = int(round(target_rate))
            original_fs_int = int(round(original_fs))

            common_div = gcd(target_rate_int, original_fs_int)
            up = target_rate_int // common_div
            down = original_fs_int // common_div

            signal = resample_poly(raw_signal, up, down)
            fs_out = float(target_rate_int)
            time = np.arange(len(signal)) / fs_out
            resampled = True
        else:
            signal = raw_signal.astype(float, copy=False)
            time = raw_time.astype(float, copy=False)
            fs_out = float(original_fs)
            resampled = False

        meta = {
            "h5_path": h5_path,
            "source_group": "resp_clean",
            "source_signal_key": "resp_clean/signal",
            "source_time_key": "resp_clean/time",
            "original_fs": float(original_fs),
            "target_rate": None if target_rate is None else float(target_rate),
            "final_fs": float(fs_out),
            "resampled": resampled,
            "n_samples": int(len(signal)),
            "duration_sec": float(time[-1] - time[0]) if len(time) > 1 else 0.0,
        }

        return signal, time, fs_out, meta

    except Exception as e:
        print(f"Error loading {h5_path}: {e}")
        return None, None, None, None

## Step 1 — Build BLA_ChR 3 s feature matrix


In [11]:
# Step 1 — Build BLA_ChR 3 s window feature matrix
# =========================================================
# This cell ONLY builds the BLA_ChR cohort.
# Cohort1 (ECG_cohort1) is appended in Step 2 below.
# =========================================================

# Build trial_id -> path mapping for the BLA_ChR directory using rename_map
rank_resp_paths = build_rank_cagemate_resp_paths_from_dir(
    h5_dir=cagemate_rank_h5s,
    rename_map=rename_map,
)

# Window-level feature matrix for BLA_ChR
rank_feature_df = build_rank_cagemate_window_feature_matrix(
    resp_paths=rank_resp_paths,
    rename_map=rename_map,
    data_type="rodentAirflow",
    target_srate=400,
    window_sec=3.0,
    n_windows_per_session=100,
    min_breaths_per_window=5,
    random_state=42,
)

# Tag provenance explicitly
rank_feature_df["SourceCohort"] = "BLA_ChR"

rank_feature_df.head()

NameError: name 'build_cagemate_rank_feature_matrix' is not defined

## Step 2 — Append cohort1 (ECG_cohort1) via trial-key rank map


In [ ]:
# Step 2 — Append cohort1 (ECG_cohort1) via trial-key rank map
# =========================================================
# This appends cohort1 rows to the Step 1 matrix.
# =========================================================

cm_rank_map = {
    "1_1": "Subordinate",
    "1_2": "Dominant",
    "2_3": "Subordinate",
    "2_4": "Dominant",
    "3_5": "Subordinate",
    "3_6": "Dominant",
    "4_7": "Subordinate",
    "4_8": "Dominant",
}

pair_groups_cm = [
    ["1_1", "1_2"],
    ["2_3", "2_4"],
    ["3_5", "3_6"],
    ["4_7", "4_8"],
]

cohort1_cm_dir = Path(
    r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs"
)

# Map short trial IDs onto full cohort1 H5 filenames on disk.
# Keys are consistent between the explicit-path builder and the
# rank-map builder so we can reuse this dict in both.
resp_paths_cm_cohort1 = {
    "CM_1_1_d1_2": str(cohort1_cm_dir / "CM_s1_1_d1_2_20250623_111352_merged.h5"),
    "CM_1_2_sub1_1": str(cohort1_cm_dir / "CM_s1_2_sub1_1_20250623_133932_merged.h5"),
    "CM_2_3_d2_4": str(cohort1_cm_dir / "CM_s2_3_d2_4_20250623_151153_merged.h5"),
    "CM_2_4_sub2_3": str(cohort1_cm_dir / "CM_s2_4_sub2_3_20250623_143348_merged.h5"),
    "CM_3_5_d3_6": str(cohort1_cm_dir / "CM_s3_5_d3_6_20250623_170708_merged.h5"),
    "CM_3_6_sub3_5": str(cohort1_cm_dir / "CM_s3_6_sub3_5_20250623_174348_merged.h5"),
    "CM_4_7_d4_8": str(cohort1_cm_dir / "CM_s4_7_d4_8_20250623_193718_merged.h5"),
    "CM_4_8_sub4_7": str(cohort1_cm_dir / "CM_s4_8_sub4_7_20250623_182649_merged.h5"),
}


def normalize_rank_label(label):
    if pd.isna(label):
        return np.nan

    label = str(label).strip().lower()
    if label in {"dom", "dominant"}:
        return "Dominant"
    if label in {"sub", "subordinate"}:
        return "Subordinate"
    return str(label).strip().title()


def parse_cm_trial_key(trial_key):
    match = re.match(r"^CM_(\d+_\d+)_([a-z]+)(\d+_\d+)$", trial_key, flags=re.IGNORECASE)
    if not match:
        return np.nan, np.nan

    return match.group(1), match.group(3)


def build_cagemate_rank_feature_matrix_from_rank_map(
    file_map,
    rank_map,
    data_type="rodentAirflow",
    target_srate=400,
    window_sec=3.0,
    n_windows_per_session=100,
    min_breaths_per_window=5,
    random_state=42,
):
    all_rows = []

    for trial_key, h5_path in file_map.items():
        subject_id, cagemate_id = parse_cm_trial_key(trial_key)
        if pd.isna(subject_id) or pd.isna(cagemate_id):
            print(f"  ⚠️ Could not parse subject IDs from {trial_key}; skipping")
            continue

        subject_rank = normalize_rank_label(rank_map.get(subject_id, np.nan))
        cagemate_rank = normalize_rank_label(rank_map.get(cagemate_id, np.nan))

        print(f"\n[Rank-map] {trial_key}")
        print(f"  subject -> {subject_id} ({subject_rank})")
        print(f"  cm      -> {cagemate_id} ({cagemate_rank})")

        signal, time, fs, load_meta = load_clean_resp_signal_cagemate_rank(
            h5_path,
            target_rate=target_srate,
        )

        if signal is None:
            print("  ❌ Load failed")
            continue

        if len(signal) == 0 or len(time) == 0:
            print("  ❌ Empty signal after preprocessing")
            continue

        bm = fit_bm_session(signal, fs, data_type=data_type)
        if bm is None:
            print("  ❌ BreathMetrics fit failed")
            continue

        n_breaths_total = len(getattr(bm, "inhaleOnsets", []))
        print(f"  ✓ {n_breaths_total} breaths detected in full session")

        windows = sample_random_nonoverlapping_windows(
            time=time,
            signal=signal,
            window_dur=window_sec,
            n_windows=n_windows_per_session,
            seed=random_state,
            allow_partial_if_short=False,
        )

        if len(windows) == 0:
            print("  ⚠️ No usable full windows")
            continue

        window_rows = extract_features_from_windows(
            bm=bm,
            windows=windows,
            min_breaths_per_window=min_breaths_per_window,
        )

        if len(window_rows) == 0:
            print("  ⚠️ No usable windows after min-breath filtering")
            continue

        print(f"  ✓ {len(window_rows)} usable windows")

        session_duration_sec = float(time[-1] - time[0]) if len(time) > 1 else np.nan
        recording_name = os.path.basename(h5_path)

        for row in window_rows:
            row.update({
                "Trial": trial_key,
                "Recording": recording_name,
                "StandardizedRecording": recording_name,
                "Subject": subject_id,
                "Cagemate": cagemate_id,
                "PairID": subject_id.split("_")[0],
                "SubjectRankCode": np.nan,
                "CagemateRankCode": np.nan,
                "SubjectRankName": subject_rank,
                "CagemateRankName": cagemate_rank,
                "Condition": subject_rank,
                "RankLabel": subject_rank,
                "Type": "CagemateRank",
                "SourceCohort": "ECG_cohort1",
                "session_duration_sec": session_duration_sec,
                "n_breaths_total": n_breaths_total,
                "WindowSec": window_sec,
                "TargetSRate": target_srate,
                "MinBreathsPerWindow": min_breaths_per_window,
            })
            all_rows.append(row)

    if not all_rows:
        print("\n⚠️ No rank-map rows collected.")
        return pd.DataFrame()

    rankmap_df = pd.DataFrame(all_rows).reset_index(drop=True)
    rankmap_df["Condition"] = rankmap_df["Condition"].map(normalize_rank_label)
    rankmap_df["RankLabel"] = rankmap_df["RankLabel"].map(normalize_rank_label)
    if "SubjectRankName" in rankmap_df.columns:
        rankmap_df["SubjectRankName"] = rankmap_df["SubjectRankName"].map(normalize_rank_label)
    if "CagemateRankName" in rankmap_df.columns:
        rankmap_df["CagemateRankName"] = rankmap_df["CagemateRankName"].map(normalize_rank_label)

    print(f"\n✅ Rank-map cohort: {len(rankmap_df)} rows × {len(rankmap_df.columns)} cols")
    print(f"   Recordings : {rankmap_df['Recording'].nunique()}")
    print(f"   Subjects   : {rankmap_df['Subject'].nunique()}")
    print(f"   Pairs      : {rankmap_df['PairID'].nunique()}")
    print(f"   Rank dist  :\n{rankmap_df['RankLabel'].value_counts(dropna=False)}")

    return rankmap_df


rank_feature_df = rank_feature_df.copy()
rank_feature_df["Condition"] = rank_feature_df["Condition"].map(normalize_rank_label)
rank_feature_df["RankLabel"] = rank_feature_df["RankLabel"].map(normalize_rank_label)
if "SubjectRankName" in rank_feature_df.columns:
    rank_feature_df["SubjectRankName"] = rank_feature_df["SubjectRankName"].map(normalize_rank_label)
if "CagemateRankName" in rank_feature_df.columns:
    rank_feature_df["CagemateRankName"] = rank_feature_df["CagemateRankName"].map(normalize_rank_label)

rank_feature_df_cm_missing = build_cagemate_rank_feature_matrix_from_rank_map(
    file_map=resp_paths_cm_cohort1,
    rank_map=cm_rank_map,
    data_type="rodentAirflow",
    target_srate=400,
    window_sec=3.0,
    n_windows_per_session=100,
    min_breaths_per_window=5,
    random_state=42,
)

rank_feature_df = pd.concat([rank_feature_df, rank_feature_df_cm_missing], ignore_index=True, sort=False)

print(f"\nCombined rank feature matrix now has {len(rank_feature_df)} rows")

if "SourceCohort" in rank_feature_df.columns:
    print("\nSourceCohort counts:")
    print(rank_feature_df["SourceCohort"].value_counts(dropna=False))

print("\nRankLabel counts:")
print(rank_feature_df["RankLabel"].value_counts(dropna=False))

## QC — minimum breaths per window


In [ ]:
# QC — minimum breaths per window
rank_feature_df['n_breaths'].min()

np.int64(5)

## QC — maximum breaths per window


In [ ]:
# QC — maximum breaths per window
rank_feature_df['n_breaths'].max()

np.int64(33)

## Save combined feature matrix to pickle


In [ ]:
# Save combined feature matrix to pickle
# pickle
rank_feature_df.to_pickle(r"C:\Users\thoma\Code\ResearchCode\respiratory_pilot\notebooks\thomas_notebooks\resp_classification\data\raw-breathmetrics-features-session-3sec-rank")